# Continuous Delivery — Build & Deploy Pipeline

## Objective
The second Level 2 notebook. NB5 added CI (test every PR); this adds CD: on merge to `main`, build a container and deploy it — to staging automatically, with the managed-eval threshold as the gate, and to production behind a manual approval.

This repo's demo triggers the whole pipeline on a PR so you can showcase it without merging; production-style CD would trigger on merge to `main`.

As with NB5, this notebook scaffolds and explains the pipeline; the pipeline runs in GitHub Actions. CD is where the move to the container/source deploy pays off: the engine gets a `sourceCodeSpec`, runtime revisions populate, and you ship the exact image you tested.

### Work Flow
Prerequisite before CD can deploy (CI in NB5 needs none of it). This custom repo wires CI/CD with Terraform — the small `cicd/` folder (not the big `terraform/` L1 config). Three steps, all runnable below:
1. Set `cicd.github_repo` in `configs/defaults.yaml` (Terraform reads it; already set to your repo).
2. `terraform -chdir=../cicd apply` → WIF (keyless auth) + Artifact Registry + the generated `pipeline.yml`.
3. Push the workflows and open a PR — `ci.yml` → `cd.yml` run on the PR (no merge needed).

Note: `agents-cli infra cicd` is the one-command alternative, but it only works on agents-cli-scaffolded projects — not this custom multi-agent repo. (Same backend: it generates the same Terraform + workflow.)

## The CD Flow

![CD flow](imgs/cd_temp.png)

- **Staging auto, production manual**: consistent with the L1 best-practices gate — automation never ships to prod without a human.
- **Eval-as-a-gate**: the managed eval from NB2 becomes a deployment gate; a regression blocks promotion.
- **Keyless auth**: GitHub authenticates to GCP via Workload Identity Federation (no service-account JSON keys).
- **Agent Identity & least-privilege**: CI lints the IAM *config* (`tests/test_identity.py` — dedicated per-agent SA, role allowlists, no long-lived keys); CD adds a described post-deploy gate that verifies the deployed agent's Agent Identity (SPIFFE) is issued and least-privilege before promoting (the runtime check is a partly-Preview surface, so it's described, not enforced).

## Set Up CI/CD — Terraform (the `cicd/` folder)

Runs the small, self-contained `cicd/` Terraform (its own state, in `cicd/`): it provisions only WIF (keyless GitHub→GCP auth) + a deployer service account + Artifact Registry, and renders the GitHub Actions orchestrator from its outputs. It does not touch the L1 `terraform/` config or re-run NB1–5.

In [15]:
# 1. Confirm config — Terraform (cicd/wif.tf) reads cicd.github_repo from configs/defaults.yaml.
import yaml
cfg = yaml.safe_load(open('../configs/defaults.yaml'))
PROJECT = cfg['project']['id']
REPO    = (cfg.get('cicd') or {}).get('github_repo', '')
assert REPO, "Set cicd.github_repo: 'owner/repo' in configs/defaults.yaml first."
print(f'project = {PROJECT}')
print(f'repo    = {REPO}')

project = sandbox-401718
repo    = behardja/agent-ops-maturity


### Auth — Two Different Credentials

- **Terraform → GCP**: uses Application Default Credentials. On a GCP notebook VM these are usually already present (the VM's service account); if not, run `gcloud auth application-default login`.
- **GitHub**: only for the `git push` of the workflow later — your `gh auth login` (as `behardja`) covers that.

Quick check:

In [16]:
!gcloud auth application-default print-access-token >/dev/null 2>&1 && echo 'ADC: ok (Terraform can auth to GCP)' || echo 'ADC: run  gcloud auth application-default login'
!gh auth status

ADC: ok (Terraform can auth to GCP)
]11;?\github.com
  ✓ Logged in to github.com account behardja (/home/jupyter/.config/gh/hosts.yml)
  - Active account: true
  - Git operations protocol: https
  - Token: gho_************************************
  - Token scopes: 'gist', 'read:org', 'repo', 'workflow'


### Provision the CI/CD infra (only `cicd/`)

`init` then `apply`. `apply` creates real resources (WIF pool/provider, deployer SA + roles, Artifact Registry) **in the `cicd/` scope only**, and **generates `.github/workflows/pipeline.yml`** from `cicd/pipeline.yml.tftpl` with this project's values (a Terraform `local_file` in `cicd/pipeline.tf`). The template ships **no** committed `pipeline.yml` — it carries project-specific values, so each clone generates its own. Swap `apply -auto-approve` for `plan` first if you want to preview.

In [17]:
!terraform -chdir=../cicd init -input=false

Initializing the backend...
Initializing provider plugins...
- Reusing previous version of hashicorp/google from the dependency lock file
- Using previously-installed hashicorp/google v7.35.0

Terraform has been successfully initialized!

You may now begin working with Terraform. Try running "terraform plan" to see
any changes that are required for your infrastructure. All Terraform commands
should now work.

If you ever set or change modules or backend configuration for Terraform,
rerun this command to reinitialize your working directory. If you forget, other
commands will detect it and remind you to do so if necessary.


In [18]:
# Real infra — scoped to the cicd/ folder (WIF + Artifact Registry). Needs ADC (above).
!terraform -chdir=../cicd apply -auto-approve

google_project_service.required["aiplatform.googleapis.com"]: Refreshing state... [id=sandbox-401718/aiplatform.googleapis.com]
google_project_service.required["iamcredentials.googleapis.com"]: Refreshing state... [id=sandbox-401718/iamcredentials.googleapis.com]
google_project_service.required["artifactregistry.googleapis.com"]: Refreshing state... [id=sandbox-401718/artifactregistry.googleapis.com]
google_project_service.required["sts.googleapis.com"]: Refreshing state... [id=sandbox-401718/sts.googleapis.com]
google_project_service.required["cloudbuild.googleapis.com"]: Refreshing state... [id=sandbox-401718/cloudbuild.googleapis.com]
google_project_service.required["iam.googleapis.com"]: Refreshing state... [id=sandbox-401718/iam.googleapis.com]
google_service_account.deployer: Refreshing state... [id=projects/sandbox-401718/serviceAccounts/agent-cd-deployer@sandbox-401718.iam.gserviceaccount.com]
google_iam_workload_identity_pool.github: Refreshing state... [id=projects/sandbox-40

### The orchestrator is generated by Terraform

`ci.yml` and `cd.yml` are **reusable workflows** already committed in `.github/workflows/` (developed independently — `ci.yml` = lint+pytest, `cd.yml` = deploy + eval gate + manual prod). The thin **`pipeline.yml`** orchestrator is **rendered by the `terraform apply` above** (`cicd/pipeline.tf` → `local_file` from `pipeline.yml.tftpl`), filling in the project / WIF-provider / SA values. It triggers on a **PR to `main`** and runs **`ci.yml` → `cd.yml`** (`cd needs ci`, so CD only runs if CI passes). The cell below just shows what was written.

In [19]:
# pipeline.yml is created by `terraform apply` (cicd/pipeline.tf -> local_file). Show it.
import pathlib
p = pathlib.Path('../.github/workflows/pipeline.yml')
print(p.read_text() if p.exists() else 'Run the terraform apply cell above first — it generates pipeline.yml.')

wrote ../.github/workflows/pipeline.yml  (PR orchestrator -> ci.yml -> cd.yml)


### Push, then open a PR to run the pipeline

`terraform apply` just **generated** `pipeline.yml` with your project's values (it isn't committed in the template — Terraform creates it, so it's a normal new file `git add` picks up). Commit the three workflow files and push, then **open a PR to `main` (don't merge)** — `pipeline.yml` runs `ci.yml` → `cd.yml` on the PR (CI gates CD; prod waits for manual approval). Re-run by pushing to the PR branch.

```bash
git add .github/workflows/{ci,cd,pipeline}.yml && git commit -m "wire CI/CD pipeline" && git push
```

Create the `staging` / `production` **Environments** in repo settings; add a *required reviewer* to `production` (you self-approve in the demo).

*TODO: add screenshot of the GitHub Actions run.*

### CI/CD is set up — push code and open a PR to run the pipeline

## Rollback, Now Unblocked

Because the CD deploy is source/container, `runtimeRevisions` populate — so the traffic-split / canary rollback described in NB4 becomes implementable: keep the prior revision and shift traffic back instead of redeploying. (Verify the `trafficSplitManual` operation on a real two-revision engine before relying on it.)

## Recap & Next

- Production-style CI/CD, PR-triggered. Reusable `ci.yml` (lint+pytest) and `cd.yml` (deploy → eval gate → manual prod) are separate files; a thin `pipeline.yml` runs them in sequence (`cd needs ci`) on a PR to `main`. Nothing fires on push.
- Infra is the `cicd/` Terraform (`wif.tf`: WIF + Artifact Registry); `pipeline.yml` is generated from its outputs. Keyless via WIF.
- Container deploy → revisions populate → real traffic-split rollback (NB4) becomes possible.

This completes the L1 → L2 arc: a tested PR (CI) that builds, deploys, and eval-gates (CD) with a manual prod approval.

Tear everything down with `07-teardown.ipynb` (also removes the CD infra: Artifact Registry, WIF pool/provider).